# 01 — Data Audit

## Urban Mobility Intelligence — NYC Yellow Taxi

Este notebook realiza una auditoría inicial de los datos de viajes de NYC Yellow Taxi correspondientes a julio de 2026.

### Objetivos
- Verificar la estructura y el tamaño del dataset.
- Revisar tipos de datos y esquema.
- Analizar valores nulos.
- Detectar posibles anomalías y valores extremos.
- Investigar registros potencialmente duplicados.
- Documentar problemas de calidad antes de realizar cualquier transformación.

> En esta etapa no se eliminan ni modifican registros. El objetivo es comprender la calidad y estructura de los datos originales.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

In [2]:
# Configurar formato numérico para facilitar la lectura
pd.options.display.float_format = "{:,.2f}".format

In [3]:
ruta = Path("../data/raw/yellow_tripdata_2026-07.parquet")

ruta.exists()

True

In [4]:
# Inspeccionar el tamaño del dataset sin cargar todas las filas en Pandas
parquet_file = pq.ParquetFile(ruta)

print("Número de filas:", parquet_file.metadata.num_rows)
print("Número de columnas:", parquet_file.metadata.num_columns)
print("Número de grupos de filas:", parquet_file.metadata.num_row_groups)

Número de filas: 3530109
Número de columnas: 21
Número de grupos de filas: 4


In [5]:
# Inspeccionar el esquema y los tipos de datos del archivo
print(parquet_file.schema)

required group field_id=-1 schema {
  optional int32 field_id=-1 VendorID;
  optional int64 field_id=-1 tpep_pickup_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 tpep_dropoff_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 passenger_count;
  optional double field_id=-1 trip_distance;
  optional int64 field_id=-1 RatecodeID;
  optional binary field_id=-1 store_and_fwd_flag (String);
  optional int32 field_id=-1 PULocationID;
  optional int32 field_id=-1 DOLocationID;
  optional int64 field_id=-1 payment_type;
  optional double field_id=-1 fare_amount;
  optional double field_id=-1 extra;
  optional double field_id=-1 mta_tax;
  optional double field_id=-1 tip_amount;
  optional double field_id=-1 tolls_amount;
  optional double field_id=-1 improvement_surcharge;
  op

In [6]:
# Cargar el primer grupo de filas como muestra para la auditoría inicial
muestra = parquet_file.read_row_group(0).to_pandas()

muestra.shape

(1048576, 21)

In [7]:
muestra.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-07-01 00:36:25,2026-07-01 00:43:18,1,2.00,1,N,138,138,2,...,8.00,0.50,0.00,0.00,1.00,19.50,0.00,2.00,0.00,None
1,1,2026-07-01 00:02:53,2026-07-01 00:17:12,1,3.20,1,N,162,112,2,...,4.25,0.50,0.00,0.00,1.00,21.35,2.50,0.00,0.75,None
2,2,2026-07-01 00:18:56,2026-07-01 00:27:38,1,3.18,1,N,137,263,1,...,1.00,0.50,4.81,0.00,1.00,24.06,2.50,0.00,0.75,None
3,1,2026-07-01 00:35:06,2026-07-01 00:57:51,0,10.00,1,N,138,246,1,...,11.25,0.50,12.30,7.46,1.00,74.01,2.50,2.00,0.75,None
4,7,2026-07-01 00:36:59,2026-07-01 00:36:59,2,9.62,1,N,140,106,1,...,0.00,0.50,9.59,0.00,1.00,57.54,2.50,0.00,0.75,None


In [8]:
# Calcular el uso de memoria de la muestra
memoria_mb = muestra.memory_usage(deep=True).sum() / 1024**2

print(f"Memoria utilizada: {memoria_mb:.2f} MB")

Memoria utilizada: 214.00 MB


In [9]:
# Revisar estructura, tipos de datos y valores no nulos
muestra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048576 entries, 0 to 1048575
Data columns (total 21 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               1048576 non-null  int32         
 1   tpep_pickup_datetime   1048576 non-null  datetime64[us]
 2   tpep_dropoff_datetime  1048576 non-null  datetime64[us]
 3   passenger_count        1048576 non-null  int64         
 4   trip_distance          1048576 non-null  float64       
 5   RatecodeID             1048576 non-null  int64         
 6   store_and_fwd_flag     1048576 non-null  object        
 7   PULocationID           1048576 non-null  int32         
 8   DOLocationID           1048576 non-null  int32         
 9   payment_type           1048576 non-null  int64         
 10  fare_amount            1048576 non-null  float64       
 11  extra                  1048576 non-null  float64       
 12  mta_tax                10485

In [10]:
# Revisar request_source en el dataset completo
request_source = parquet_file.read(
    columns=["request_source"]
).to_pandas()

request_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3530109 entries, 0 to 3530108
Data columns (total 1 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   request_source  object
dtypes: object(1)
memory usage: 26.9+ MB


In [11]:
# Contar valores nulos en request_source
request_source["request_source"].isna().sum()

np.int64(2560692)

In [12]:
# Revisar la distribución de valores de request_source
request_source["request_source"].value_counts(dropna=False)

request_source
None      2560692
HV0003     739048
A          223309
EH0004       6409
CC            642
EH0010          9
Name: count, dtype: int64

In [13]:
# Calcular la distribución porcentual de request_source
request_source["request_source"].value_counts(
    normalize=True,
    dropna=False
) * 100

request_source
None     72.54
HV0003   20.94
A         6.33
EH0004    0.18
CC        0.02
EH0010    0.00
Name: proportion, dtype: float64

In [14]:
# Calcular valores nulos por columna en el dataset completo
tabla_nulos = []

for i in range(parquet_file.metadata.num_columns):
    nombre = parquet_file.schema.names[i]
    
    total_nulos = sum(
        parquet_file.metadata.row_group(rg)
        .column(i)
        .statistics
        .null_count
        for rg in range(parquet_file.metadata.num_row_groups)
    )
    
    tabla_nulos.append([nombre, total_nulos])

pd.DataFrame(
    tabla_nulos,
    columns=["columna", "nulos"]
)

,columna,nulos
0,VendorID,0
1,tpep_pickup_datetime,0
2,tpep_dropoff_datetime,0
3,passenger_count,969727
4,trip_distance,0
5,RatecodeID,969727
6,store_and_fwd_flag,969727
7,PULocationID,0
8,DOLocationID,0
9,payment_type,0


In [15]:
# Calcular el porcentaje de valores nulos por columna
df_nulos = pd.DataFrame(
    tabla_nulos,
    columns=["columna", "nulos"]
)

df_nulos["porcentaje_nulos"] = (
    df_nulos["nulos"] / parquet_file.metadata.num_rows * 100
).round(2)

df_nulos

,columna,nulos,porcentaje_nulos
0,VendorID,0,0.00
1,tpep_pickup_datetime,0,0.00
2,tpep_dropoff_datetime,0,0.00
3,passenger_count,969727,27.47
4,trip_distance,0,0.00
5,RatecodeID,969727,27.47
6,store_and_fwd_flag,969727,27.47
7,PULocationID,0,0.00
8,DOLocationID,0,0.00
9,payment_type,0,0.00


In [16]:
# Analizar el patrón conjunto de nulos en cinco columnas
columnas_patron = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

patron_nulos = parquet_file.read(
    columns=columnas_patron
).to_pandas()

In [17]:
# Contar filas donde las cinco columnas son nulas simultáneamente
patron_nulos.isna().all(axis=1).sum()

np.int64(969727)

In [18]:
# Identificar las filas que pertenecen al patrón de nulos
patron_nulos["grupo_nulos"] = patron_nulos[
    "passenger_count"
].isna()

patron_nulos["grupo_nulos"].value_counts()

grupo_nulos
False    2560382
True      969727
Name: count, dtype: int64

### Hallazgo

- **2.560.382 viajes:** las cinco variables contienen datos.
- **969.727 viajes:** las cinco variables presentan valores nulos simultáneamente.

Esto indica que los valores faltantes siguen un patrón conjunto y no aparecen de forma aislada.

In [19]:
# Comparar el patrón de nulos con request_source
patron_nulos["request_source"] = request_source["request_source"]

pd.crosstab(
    patron_nulos["grupo_nulos"],
    patron_nulos["request_source"],
    dropna=False
)

request_source,A,CC,EH0004,EH0010,HV0003,NaN
grupo_nulos,,,,,,
False,0,0,0,0,0,2560382
True,223309,642,6409,9,739048,310


### Hallazgo

De los **969.727 registros** que presentan el patrón conjunto de nulos, **969.417 tienen `request_source` informado** y solo **310 registros** también presentan `request_source` nulo.

Esto muestra una fuerte relación entre el patrón de nulos y la presencia de información en `request_source`.

In [20]:
# Comparar el patrón de nulos con el tipo de pago
payment = parquet_file.read(
    columns=["payment_type"]
).to_pandas()

patron_nulos["payment_type"] = payment["payment_type"]

pd.crosstab(
    patron_nulos["grupo_nulos"],
    patron_nulos["payment_type"]
)

payment_type,0,1,2,3,4
grupo_nulos,,,,,
False,0,2182461,345646,10893,21382
True,969727,0,0,0,0


### Hallazgo — Relación con `payment_type`

El **100% de las 969.727 filas** que presentan el patrón conjunto de cinco variables nulas tiene `payment_type = 0`.

Según el diccionario de datos de NYC TLC, `payment_type = 0` corresponde a **Flex Fare trips**.

En este dataset se observa una asociación exacta entre `payment_type = 0` y el patrón de nulos analizado:

- **2.560.382 registros:** `payment_type` entre 1 y 4 y variables tradicionales presentes.
- **969.727 registros:** `payment_type = 0`, las cinco variables analizadas son nulas y `request_source` está informado en casi todos los casos.

Por lo tanto, estos valores nulos presentan un patrón estructurado asociado a los registros Flex Fare y no deben eliminarse automáticamente durante la limpieza.

In [21]:
# Analizar valores cero, negativos y extremos en trip_distance

distancia = parquet_file.read(
    columns=["trip_distance"]
).to_pandas()

distancia["trip_distance"].describe()

count   3,530,109.00
mean            5.55
std           576.20
min             0.00
25%             1.03
50%             1.90
75%             3.94
max       318,129.10
Name: trip_distance, dtype: float64

In [22]:
# Cuantificar distancias iguales a cero y valores extremos
print("Distancia = 0:",
      (distancia["trip_distance"] == 0).sum())

print("Distancia < 0:",
      (distancia["trip_distance"] < 0).sum())

print("Distancia > 100:",
      (distancia["trip_distance"] > 100).sum())

print("Distancia > 1000:",
      (distancia["trip_distance"] > 1000).sum())


Distancia = 0: 128322
Distancia < 0: 0
Distancia > 100: 172
Distancia > 1000: 61


In [23]:
# Revisar las 10 distancias más altas registradas
distancia.nlargest(10, "trip_distance")

,trip_distance
3458861,"318,129.10"
3208119,"317,017.94"
3414440,"270,530.29"
3209680,"247,657.16"
2637969,"217,780.00"
3273511,"189,585.48"
2917549,"188,214.94"
2715885,"188,030.58"
2987590,"181,291.03"
3361769,"178,321.52"


In [24]:
# Cargar variables relevantes para investigar distancias extremas
columnas_revision = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "total_amount",
    "request_source"
]

revision = parquet_file.read(
    columns=columnas_revision
).to_pandas()


In [25]:
# Inspeccionar los 10 registros con mayor distancia
revision.nlargest(10, "trip_distance")

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,total_amount,request_source
3458861,2026-07-29 22:02:00,2026-07-29 22:33:00,"318,129.10",148,28,0,44.18,48.93,HV0003
3208119,2026-07-21 20:32:00,2026-07-21 20:56:00,"317,017.94",87,143,0,38.19,42.94,HV0003
3414440,2026-07-28 08:47:00,2026-07-28 09:20:00,"270,530.29",66,68,0,65.91,70.66,HV0003
3209680,2026-07-21 21:22:00,2026-07-21 21:38:00,"247,657.16",163,90,0,26.10,30.85,HV0003
2637969,2026-07-03 20:07:00,2026-07-03 20:20:00,"217,780.00",137,100,0,16.71,21.46,HV0003
3273511,2026-07-23 22:14:00,2026-07-23 22:26:00,"189,585.48",107,162,0,22.09,26.84,HV0003
2917549,2026-07-13 17:59:00,2026-07-13 18:19:00,"188,214.94",151,119,0,20.44,21.94,HV0003
2715885,2026-07-06 17:23:00,2026-07-06 17:31:00,"188,030.58",24,142,0,13.52,17.52,HV0003
2987590,2026-07-16 06:11:00,2026-07-16 06:27:00,"181,291.03",13,140,0,21.15,25.90,HV0003
3361769,2026-07-26 05:29:00,2026-07-26 05:37:00,"178,321.52",142,246,0,22.69,27.44,HV0003


In [26]:
# Comparar distancias extremas (>100 millas) por tipo de pago
pd.crosstab(
    revision["payment_type"],
    revision["trip_distance"] > 100
)

trip_distance,False,True
payment_type,,
0,969620,107
1,2182431,30
2,345614,32
3,10893,0
4,21379,3


In [27]:
# Comparar viajes con distancia 0 por tipo de pago
pd.crosstab(
    revision["payment_type"],
    revision["trip_distance"] == 0
)

trip_distance,False,True
payment_type,,
0,878362,91365
1,2159706,22755
2,336757,8889
3,8051,2842
4,18911,2471


In [28]:
# Calcular el porcentaje de viajes con distancia 0 por tipo de pago
porcentaje_cero = (
    revision
    .groupby("payment_type")["trip_distance"]
    .apply(lambda x: (x == 0).mean() * 100)
    .round(2)
)

porcentaje_cero

payment_type
0    9.42
1    1.04
2    2.57
3   26.09
4   11.56
Name: trip_distance, dtype: float64

### Hallazgo — Viajes con distancia 0

La proporción de viajes con distancia 0 varía según el tipo de pago:

- `payment_type = 0`: ~9%
- `payment_type = 1`: ~1%
- `payment_type = 2`: ~3%
- `payment_type = 3`: **26,09%**
- `payment_type = 4`: ~12%

Aunque `payment_type = 0` presenta la mayor cantidad absoluta de casos (91.365), `payment_type = 3` tiene la mayor proporción de viajes con distancia 0 (26,09%).

Por lo tanto, es importante distinguir entre **cantidad absoluta** y **proporción dentro de cada grupo**.


In [29]:
# Analizar la duración de los viajes con distancia 0
viajes_cero = revision[
    revision["trip_distance"] == 0
].copy()

viajes_cero["duracion_min"] = (
    viajes_cero["tpep_dropoff_datetime"]
    - viajes_cero["tpep_pickup_datetime"]
).dt.total_seconds() / 60

viajes_cero["duracion_min"].describe()

count   128,322.00
mean         16.25
std          27.82
min           0.00
25%           3.92
50%          14.93
75%          23.77
max       4,253.85
Name: duracion_min, dtype: float64

In [30]:
# Cuantificar anomalías de duración en viajes con distancia 0
print("Duración = 0:",
      (viajes_cero["duracion_min"] == 0).sum())

print("Duración < 0:",
      (viajes_cero["duracion_min"] < 0).sum())

print("Duración > 60 min:",
      (viajes_cero["duracion_min"] > 60).sum())

print("Duración > 180 min:",
      (viajes_cero["duracion_min"] > 180).sum())


Duración = 0: 1251
Duración < 0: 0
Duración > 60 min: 1100
Duración > 180 min: 34


In [31]:
# Inspeccionar los viajes con distancia 0 y mayor duración
viajes_cero.nlargest(10, "duracion_min")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "duracion_min",
        "payment_type",
        "fare_amount",
        "total_amount",
        "request_source"
    ]
]

,tpep_pickup_datetime,tpep_dropoff_datetime,duracion_min,payment_type,fare_amount,total_amount,request_source
428139,2026-07-07 10:36:40,2026-07-10 09:30:31,"4,253.85",2,3.00,4.50,None
2196545,2026-07-28 00:52:49,2026-07-30 16:34:18,"3,821.48",2,70.00,74.00,None
1394987,2026-07-18 13:09:19,2026-07-20 15:10:19,"3,001.00",1,20.07,24.07,None
3081312,2026-07-18 13:09:19,2026-07-20 15:10:19,"3,001.00",0,20.07,24.07,EH0004
2388824,2026-07-30 08:52:12,2026-08-01 09:04:23,"2,892.18",2,3.00,7.00,None
2114168,2026-07-26 23:34:42,2026-07-28 01:34:59,"1,560.28",4,38.62,40.12,None
35160,2026-07-01 14:30:16,2026-07-02 14:21:24,"1,431.13",1,8.60,14.69,None
2464207,2026-07-30 21:43:57,2026-07-31 19:32:12,"1,308.25",2,800.00,801.00,None
279938,2026-07-05 00:22:00,2026-07-05 20:04:10,"1,182.17",4,3.00,5.50,None
703255,2026-07-10 14:19:33,2026-07-11 08:46:21,"1,106.80",2,3.70,5.20,None


In [32]:
# Identificar posibles duplicados según las características principales del viaje
columnas_comparacion = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount"
]

duplicados_parciales = revision[
    revision.duplicated(
        subset=columnas_comparacion,
        keep=False
    )
]

duplicados_parciales.shape

(27090, 9)

In [33]:
# Cargar el dataset completo y recuperar las 21 variables
datos_completos = parquet_file.read().to_pandas()

sospechosos_completos = datos_completos.loc[
    duplicados_parciales.index
].copy()

sospechosos_completos.shape

(27090, 21)

In [34]:
# Verificar si existen duplicados exactos entre los registros sospechosos
duplicados_exactos = sospechosos_completos.duplicated(keep=False)

print(
    "Filas involucradas en duplicados exactos:",
    duplicados_exactos.sum()
)

Filas involucradas en duplicados exactos: 0


In [35]:
# Verificar cuántos registros forman cada grupo de viajes similares
tamanos_grupos = (
    sospechosos_completos
    .groupby(columnas_comparacion, dropna=False)
    .size()
)

tamanos_grupos.value_counts().sort_index()

2    13545
Name: count, dtype: int64

In [36]:
# Crear un identificador único para cada par de registros similares
sospechosos_completos["grupo_id"] = (
    sospechosos_completos
    .groupby(columnas_comparacion, dropna=False)
    .ngroup()
)

sospechosos_completos["grupo_id"].nunique()

13545

In [37]:
# Identificar las columnas que pueden presentar diferencias entre los pares
columnas_a_comparar = [
    col for col in sospechosos_completos.columns
    if col not in columnas_comparacion + ["grupo_id"]
]

# Ordenar y separar los dos registros de cada par
pares_ordenados = sospechosos_completos.sort_values("grupo_id")

primera_fila = pares_ordenados.iloc[0::2].reset_index(drop=True)
segunda_fila = pares_ordenados.iloc[1::2].reset_index(drop=True)

print("Pares preparados para comparar:", len(primera_fila))

Pares preparados para comparar: 13545


In [38]:
# Comparar los pares sospechosos e identificar en qué columnas presentan diferencias
cambios_por_columna = {}

for col in columnas_a_comparar:
    a = primera_fila[col].fillna("__NULO__")
    b = segunda_fila[col].fillna("__NULO__")
    
    cambios_por_columna[col] = (a != b).sum()

cambios_por_columna = (
    pd.Series(cambios_por_columna)
    .sort_values(ascending=False)
)

cambios_por_columna

passenger_count          13545
RatecodeID               13545
store_and_fwd_flag       13545
payment_type             13545
congestion_surcharge     13545
Airport_fee              13545
request_source           13378
extra                    12825
tip_amount                   9
mta_tax                      3
improvement_surcharge        3
VendorID                     0
tolls_amount                 0
cbd_congestion_fee           0
dtype: int64

In [39]:
# Comparar el tipo de pago entre los dos registros de cada par
pd.crosstab(
    primera_fila["payment_type"],
    segunda_fila["payment_type"]
)

payment_type,0,1,2,3,4
payment_type,,,,,
0,0,6719,21,9,14
1,6741,0,0,0,0
2,15,0,0,0,0
3,12,0,0,0,0
4,14,0,0,0,0


In [40]:
# Verificar el patrón de cinco variables nulas en los registros payment_type = 0
columnas_nulos = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

pares_tipo0 = sospechosos_completos[
    sospechosos_completos["payment_type"] == 0
]

pares_tipo0[columnas_nulos].isna().sum()

passenger_count         13545
RatecodeID              13545
store_and_fwd_flag      13545
congestion_surcharge    13545
Airport_fee             13545
dtype: int64

### Hallazgo — Registros similares

Se identificaron **27.090 registros** que coinciden en las principales características del viaje, formando **13.545 pares**.

Sin embargo, no se encontraron duplicados exactos.

En cada par:
- Un registro presenta `payment_type = 0`.
- El otro presenta `payment_type` entre 1 y 4.
- Los registros con `payment_type = 0` presentan valores nulos en `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee`.
- También se observan diferencias frecuentes en `request_source` y `extra`.

Por lo tanto, estos registros presentan un **patrón estructurado de diferencias** y no deben eliminarse automáticamente como duplicados.

In [41]:
# Inspeccionar una muestra de los posibles duplicados
duplicados_parciales[
    columnas_comparacion +
    ["payment_type", "request_source"]
].sort_values(
    ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
).head(20)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,fare_amount,total_amount,payment_type,request_source
905967,2026-07-13 00:01:27,2026-07-13 00:29:18,10.30,138,140,40.80,70.16,1,None
2904598,2026-07-13 00:01:27,2026-07-13 00:29:18,10.30,138,140,40.80,70.16,0,A
905546,2026-07-13 00:02:52,2026-07-13 00:07:12,0.80,79,224,5.80,13.28,1,None
2904362,2026-07-13 00:02:52,2026-07-13 00:07:12,0.80,79,224,5.80,13.28,0,A
905351,2026-07-13 00:07:16,2026-07-13 00:29:11,0.00,37,37,13.50,16.00,2,None
2904240,2026-07-13 00:07:16,2026-07-13 00:29:11,0.00,37,37,13.50,16.00,0,A
905862,2026-07-13 00:34:30,2026-07-13 01:05:26,18.10,132,158,70.00,76.75,1,None
2904535,2026-07-13 00:34:30,2026-07-13 01:05:26,18.10,132,158,70.00,76.75,0,A
906165,2026-07-13 00:40:25,2026-07-13 00:54:12,2.70,114,161,19.35,24.10,1,None
2904661,2026-07-13 00:40:25,2026-07-13 00:54:12,2.70,114,161,19.35,24.10,0,EH0004


In [42]:
# Guardar los índices de los posibles duplicados para analizarlos
indices_sospechosos = duplicados_parciales.index

len(indices_sospechosos)

27090

In [43]:
# Verifamos si estan duplicadas las filas
duplicados_exactos = sospechosos_completos.duplicated(
    keep=False
)

print(
    "Filas involucradas en duplicados exactos:",
    duplicados_exactos.sum()
)

Filas involucradas en duplicados exactos: 0


In [44]:
# Verifamos si hay similitud entre 2 pares o mas
tamanos_grupos = (
    sospechosos_completos
    .groupby(columnas_comparacion, dropna=False)
    .size()
)


tamanos_grupos.value_counts().sort_index()

2    13545
Name: count, dtype: int64

### Hallazgo — Posibles duplicados

Las **27.090 filas sospechosas** corresponden a **13.545 pares de registros** que coinciden en las características principales del viaje.

Sin embargo, **ningún par es idéntico en las 21 variables originales**, por lo que no deben considerarse duplicados exactos ni eliminarse automáticamente.

In [45]:
# Creamos un identficador para cada par
sospechosos_completos["grupo_id"] = (
    sospechosos_completos
    .groupby(columnas_comparacion, dropna=False)
    .ngroup()
)

sospechosos_completos[
    ["grupo_id"] + columnas_comparacion
].head(10)

,grupo_id,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,fare_amount,total_amount
905351,2,2026-07-13 00:07:16,2026-07-13 00:29:11,0.00,37,37,13.50,16.00
905546,1,2026-07-13 00:02:52,2026-07-13 00:07:12,0.80,79,224,5.80,13.28
905808,5,2026-07-13 00:41:27,2026-07-13 00:47:53,1.40,163,229,8.60,16.50
905862,3,2026-07-13 00:34:30,2026-07-13 01:05:26,18.10,132,158,70.00,76.75
905967,0,2026-07-13 00:01:27,2026-07-13 00:29:18,10.30,138,140,40.80,70.16
905968,6,2026-07-13 00:43:57,2026-07-13 00:52:46,1.40,186,163,9.30,17.30
906165,4,2026-07-13 00:40:25,2026-07-13 00:54:12,2.70,114,161,19.35,24.10
906721,7,2026-07-13 00:55:31,2026-07-13 01:10:25,0.00,37,61,14.50,17.00
907357,8,2026-07-13 02:20:14,2026-07-13 03:02:19,0.00,41,265,71.50,74.00
907451,9,2026-07-13 02:48:02,2026-07-13 02:53:57,1.20,107,144,12.24,16.99


In [46]:
# Identificamos las columnas
columnas_a_comparar = [
    col for col in sospechosos_completos.columns
    if col not in columnas_comparacion + ["grupo_id"]
]

columnas_a_comparar

['VendorID',
 'passenger_count',
 'RatecodeID',
 'store_and_fwd_flag',
 'payment_type',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'request_source']

In [47]:
pares_ordenados = sospechosos_completos.sort_values("grupo_id")

primera_fila = pares_ordenados.iloc[0::2].reset_index(drop=True)
segunda_fila = pares_ordenados.iloc[1::2].reset_index(drop=True)

print(primera_fila.shape)
print(segunda_fila.shape)

(13545, 22)
(13545, 22)


In [48]:
# Comparar los pares sospechosos e identificar en qué columnas presentan diferencias
cambios_por_columna = {}

for col in columnas_a_comparar:
    a = primera_fila[col].fillna("__NULO__")
    b = segunda_fila[col].fillna("__NULO__")
    
    cambios_por_columna[col] = (a != b).sum()

cambios_por_columna = (
    pd.Series(cambios_por_columna)
    .sort_values(ascending=False)
)

cambios_por_columna

passenger_count          13545
RatecodeID               13545
store_and_fwd_flag       13545
payment_type             13545
congestion_surcharge     13545
Airport_fee              13545
request_source           13378
extra                    12825
tip_amount                   9
mta_tax                      3
improvement_surcharge        3
VendorID                     0
tolls_amount                 0
cbd_congestion_fee           0
dtype: int64

In [49]:
# Comparar el tipo de pago entre los registros de cada par sospechoso
pd.crosstab(
    primera_fila["payment_type"],
    segunda_fila["payment_type"]
)

payment_type,0,1,2,3,4
payment_type,,,,,
0,0,6719,21,9,14
1,6741,0,0,0,0
2,15,0,0,0,0
3,12,0,0,0,0
4,14,0,0,0,0


In [50]:
# Filtrar los registros con payment_type = 0 dentro de los pares sospechosos
pares_tipo0 = sospechosos_completos[
    sospechosos_completos["payment_type"] == 0
]

pares_tipo0.shape

(13545, 22)

In [51]:
# Verificar el patrón de cinco variables nulas en los registros payment_type = 0
columnas_nulos = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

pares_tipo0[columnas_nulos].isna().sum()

passenger_count         13545
RatecodeID              13545
store_and_fwd_flag      13545
congestion_surcharge    13545
Airport_fee             13545
dtype: int64

## Conclusiones de la Auditoría de Calidad de Datos

El dataset de Yellow Taxi correspondiente a julio de 2026 contiene **3.530.109 registros y 21 variables**.

### Principales hallazgos

- Se detectaron valores nulos que presentan patrones estructurados y no parecen estar distribuidos aleatoriamente.
- Las columnas `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee` contienen **969.727 valores nulos cada una**.
- La columna `request_source` contiene **2.560.692 valores nulos (72,54 %)**.
- Se detectaron **128.322 registros con `trip_distance = 0`**.
- No se encontraron distancias negativas.
- Se detectaron **172 registros con distancias superiores a 100 millas**.
- De ellos, **61 registros superan las 1.000 millas**, incluyendo valores extremos claramente incompatibles con la duración registrada del viaje.
- Algunos registros con distancia 0 presentan duraciones extremadamente elevadas, por lo que requieren un tratamiento específico durante la limpieza.
- Se identificaron **27.090 registros potencialmente relacionados** al comparar fecha y hora, distancia, ubicación de origen y destino, tarifa y monto total.
- Estos registros forman **13.545 pares**.
- Al comparar las **21 variables originales**, no se detectaron duplicados exactos.
- En cada uno de los 13.545 pares aparece un registro con `payment_type = 0`.
- Dentro de estos pares, los registros con `payment_type = 0` presentan sistemáticamente valores nulos en varias variables relacionadas con el viaje.

### Decisión de calidad de datos

Los registros potencialmente anómalos **no serán eliminados automáticamente**.

Las reglas de limpieza se definirán en una etapa separada según el significado y utilidad analítica de cada variable.

Los datos originales permanecerán sin modificaciones y las transformaciones se realizarán mediante un proceso de limpieza reproducible.

## Resumen de la Auditoría

| Control de calidad | Resultado |
|---|---:|
| Total de registros | 3.530.109 |
| Total de variables | 21 |
| Duplicados exactos detectados | 0 |
| Registros con distancia igual a 0 | 128.322 |
| Registros con distancia negativa | 0 |
| Registros con distancia superior a 100 millas | 172 |
| Registros con distancia superior a 1.000 millas | 61 |
| Pares de registros similares investigados | 13.545 |
| Valores nulos en `request_source` | 72,54 % |
| Valores nulos en 5 variables relacionadas | 969.727 cada una |